In [22]:
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 150
MinUtilTime_2 = 150
MaxUtilTime = 1000

vehicle_fixed_cost = 1000
freq_penalty = 1000

nvehicle = 150
K = range(nvehicle)
H = range(HOURS)

bigM = 1440

freq1 = [0]*12 + [1, 2, 1, 2, 3, 3, 4, 5, 2, 3, 1, 0]  # ATB
freq2 = [0]*12 + [1, 2, 1, 3, 3 ,3, 5, 2, 3, 4, 1, 0]  # HSK
# freq1 = [0, 0, 0, 0, 4, 4, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 0, 0, 1, 1, 2, 2, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 4, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 0, 0, 1, 1, 2, 2, 0, 0]
# freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB freq per hour
# freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0] 

arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {("ATB", "HSK"): 110, ("HSK", "ATB"): 110}

nodes = []
node_id = 1
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

for hour in H:
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1
    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'})
end_node_id = node_id
node_id += 1

nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

E = []
synthetic_id = node_id
synthetic_nodes = []

for node in nodes:
    loc = node['loc']
    dst_time = node['time']
    if loc in ['ATB', 'HSK']:
        cost, N_bus = arc_data[('X', loc)]
        if dst_time >= cost:
            E.append({
                'src': {'id': start_node_id, 'time': 0, 'loc': 'X'},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })

for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time + travel_time[(curr_loc, transitions[curr_loc])] <= MinUtilTime_2:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_nodes.append(dst)

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        curr_src_id = synthetic_id
        curr_time = next_time
        curr_loc = next_loc
        synthetic_id += 1
        total_time += cost

    if curr_time <= MINUTES_IN_DAY:
        cost_to_X, N_bus_to_X = arc_data[(curr_loc, 'X')]
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'},
            'cost': cost_to_X,
            'N_bus': N_bus_to_X
        })

V.extend(synthetic_nodes)

for node in V:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# --- ADD WAITING AND DEADHEADING ARCS ---
MAX_WAIT = 60
augmented_E = E.copy()
unique_keys = set()

# Add existing arcs to unique_keys to avoid duplication later
for e in E:
    arc_key = (
        e['src']['id'], e['src']['time'], e['src']['loc'],
        e['dst']['id'], e['dst']['time'], e['dst']['loc'],
        e['cost'], e['N_bus']
    )
    unique_keys.add(arc_key)

# Waiting arcs
# for src in V:
#     for dst in V:
#         if src['id'] == dst['id']:
#             continue
#         time_diff = dst['time'] - src['time']
#         if src['loc'] == dst['loc'] and 0 < time_diff <= MAX_WAIT:
#             arc_key = (
#                 src['id'], src['time'], src['loc'],
#                 dst['id'], dst['time'], dst['loc'],
#                 200, 150
#             )
#             if arc_key not in unique_keys:
#                 unique_keys.add(arc_key)
#                 augmented_E.append({
#                     'src': src,
#                     'dst': dst,
#                     'cost': 200,
#                     'N_bus': 150
#                 })
# Sort nodes by location and time to easily find next closest at same loc
from collections import defaultdict

# Group nodes by location
nodes_by_loc = defaultdict(list)
for node in V:
    nodes_by_loc[node['loc']].append(node)

# Sort nodes by time for each location
for loc in nodes_by_loc:
    nodes_by_loc[loc].sort(key=lambda n: n['time'])

# Create minimal waiting arcs (only to immediate next time at same loc)
for loc, nodes in nodes_by_loc.items():
    for i in range(len(nodes) - 1):
        src = nodes[i]
        dst = nodes[i + 1]

        time_diff = dst['time'] - src['time']
        if 0 < time_diff <= MAX_WAIT:
            arc_key = (
                src['id'], src['time'], src['loc'],
                dst['id'], dst['time'], dst['loc'],
                200, 150
            )
            if arc_key not in unique_keys:
                unique_keys.add(arc_key)
                augmented_E.append({
                    'src': src,
                    'dst': dst,
                    'cost': 200,
                    'N_bus': 150
                })


# Deadheading arcs
# for src in V:
#     for dst in V:
#         if src['id'] == dst['id'] or src['loc'] == dst['loc']:
#             continue

#         key = (src['loc'], dst['loc'])
#         if key not in arc_data:
#             continue

#         travel_t, cap = arc_data[key]
#         if dst['time'] > src['time'] + travel_t:
#             arc_key = (
#                 src['id'], src['time'], src['loc'],
#                 dst['id'], dst['time'], dst['loc'],
#                 travel_t, cap
#             )
#             if arc_key not in unique_keys:
#                 unique_keys.add(arc_key)
#                 augmented_E.append({
#                     'src': src,
#                     'dst': dst,
#                     'cost': 180,
#                     'N_bus': 150
#                 })
for src in V:
    best_arcs_by_dst_loc = {}
    
    for dst in V:
        if src['id'] == dst['id'] or src['loc'] == dst['loc']:
            continue

        key = (src['loc'], dst['loc'])
        if key not in arc_data:
            continue

        travel_t, cap = arc_data[key]

        if dst['time'] > src['time'] + travel_t:
            time_diff = dst['time'] - src['time'] - travel_t

            # Keep only the best arc (least time diff) per dst location
            if dst['loc'] not in best_arcs_by_dst_loc or time_diff < best_arcs_by_dst_loc[dst['loc']]['time_diff']:
                best_arcs_by_dst_loc[dst['loc']] = {
                    'dst': dst,
                    'travel_t': travel_t,
                    'cap': cap,
                    'time_diff': time_diff
                }

    # Now select the single best arc among all dst_locs (least time_diff)
    if best_arcs_by_dst_loc:
        # Find arc with minimal time_diff overall
        best_dst_loc = min(best_arcs_by_dst_loc, key=lambda loc: best_arcs_by_dst_loc[loc]['time_diff'])
        best = best_arcs_by_dst_loc[best_dst_loc]

        arc_key = (
            src['id'], src['time'], src['loc'],
            best['dst']['id'], best['dst']['time'], best['dst']['loc'],
            best['travel_t'], best['cap']
        )

        if arc_key not in unique_keys:
            unique_keys.add(arc_key)
            augmented_E.append({
                'src': src,
                'dst': best['dst'],
                'cost': 220,
                'N_bus': 150
            })
# depot_return_arcs = []
# X = 'X'  # Depot

# for e in augmented_E:
#     if e['src']['loc'] == X or e['dst']['loc'] == X:
#         continue  # Skip depot arcs

#     src = e['src']
#     dst = e['dst']
#     direct_cost = e['cost']
#     src_to_depot_key = (src['loc'], X)
#     depot_to_dst_key = (X, dst['loc'])

#     if src_to_depot_key in arc_data and depot_to_dst_key in arc_data:
#         src_to_depot_cost, _ = arc_data[src_to_depot_key]
#         depot_to_dst_cost, _ = arc_data[depot_to_dst_key]

#         time_to_depot = src_to_depot_cost
#         time_from_depot = depot_to_dst_cost
#         total_depot_time = time_to_depot + time_from_depot

#         if dst['time'] >= src['time'] + total_depot_time:
#             depot_total_cost = src_to_depot_cost + depot_to_dst_cost

#             if depot_total_cost < direct_cost:
#                 arc_key = (
#                     src['id'], src['time'], src['loc'],
#                     dst['id'], dst['time'], dst['loc'],
#                     depot_total_cost, e['N_bus']
#                 )
#                 if arc_key not in unique_keys:
#                     unique_keys.add(arc_key)
#                     depot_return_arcs.append({
#                         'src': src,
#                         'dst': dst,
#                         'cost': depot_total_cost,
#                         'N_bus': e['N_bus']
#                     })
#                 # Remove the direct expensive arc
#                 augmented_E.remove(e)

# augmented_E.extend(depot_return_arcs)


# Final cleaned arcs
E = augmented_E
import random

# Function to build a random valid route from start to end node using arcs from E
def build_random_route(E, start_id, end_id):
    route = []
    current_id = start_id

    # Create a fast lookup for arcs by source node ID
    arc_map = {}
    for arc in E:
        src = arc['src']['id']
        arc_map.setdefault(src, []).append(arc)

    while current_id != end_id:
        options = arc_map.get(current_id, [])
        if not options:
            break  # Dead end

        arc = random.choice(options)
        route.append(arc)
        current_id = arc['dst']['id']

        if len(route) > 100:
            break

    return route if current_id == end_id else []

# Generate initial population
initial_population = []
population_size = 20  # You can increase this as needed

for _ in range(population_size):
    chromosome = build_random_route(E, start_node_id, end_node_id)
    if chromosome:
        initial_population.append(chromosome)
def print_chromosome(route, route_id=None):
    if route_id is not None:
        print(f"\nChromosome #{route_id}")
    else:
        print("\nChromosome:")

    total_cost = 0
    for i, arc in enumerate(route):
        src = arc['src']
        dst = arc['dst']
        cost = arc['cost']
        total_cost += cost
        print(f"{i+1:02d}. {src['loc']} ({src['time']:04d} min) → {dst['loc']} ({dst['time']:04d} min) | Cost: {cost}")

    print(f"Total Route Cost: {total_cost}")

# --- FINAL OUTPUT [unchanged] ---
for arc in E:
    print({
        'src_id': arc['src']['id'],
        'src_loc': arc['src']['loc'],
        'src_time': arc['src']['time'],
        'dst_id': arc['dst']['id'],
        'dst_loc': arc['dst']['loc'],
        'dst_time': arc['dst']['time'],
        'cost': arc['cost'],
        'N_bus': arc['N_bus']
    })

# Display all chromosomes in the initial population
for idx, chrom in enumerate(initial_population):
    print_chromosome(chrom, route_id=idx + 1)
# ------------------- ADD FITNESS FUNCTION HERE -------------------
MinUtilTime = 600  # in minutes, e.g., 10 hours
util_penalty = 1000  # penalty per underutilized hour
freq_penalty = 1000  # penalty per frequency shortfall

from datetime import datetime

def parse_time(tstr):
    if isinstance(tstr, int):
        hours = tstr // 60
        minutes = tstr % 60
        if hours == 24:
            hours = 0
        tstr = f"{hours:02d}:{minutes:02d}"
    elif isinstance(tstr, str) and tstr.strip() == "24:00":
        tstr = "00:00"
    return datetime.strptime(tstr, "%H:%M")

def time_diff(start_str, end_str):
    t1 = parse_time(start_str)
    t2 = parse_time(end_str)
    diff = (t2 - t1).total_seconds() / 60
    if diff < 0:
        diff += 1440
    return diff

def evaluate_fitness(route):
    total_cost = 0
    total_duration = 0

    freq_count = {
        'ATB': [0]*24,
        'HSK': [0]*24
    }

    for arc in route:
        total_cost += arc['cost']
        dst = arc['dst']
        if dst['loc'] in freq_count:
            hour = int(parse_time(dst['time']).hour)
            freq_count[dst['loc']][hour] += 1

    if route:
        start_time = route[0]['src']['time']
        end_time = route[-1]['dst']['time']
        total_duration = time_diff(start_time, end_time)

    util_penalty_cost = 0
    if total_duration < MinUtilTime:
        util_penalty_cost = util_penalty * ((MinUtilTime - total_duration) / 60)

    freq_penalty_cost = 0
    desired_freq = 1
    for terminal in freq_count:
        for hour in range(24):
            if freq_count[terminal][hour] < desired_freq:
                freq_penalty_cost += freq_penalty * (desired_freq - freq_count[terminal][hour])

    fitness = - (total_cost + util_penalty_cost + freq_penalty_cost)
    return fitness
    print("\n=== FITNESS SCORES ===")
fitness_scores = []
top_N = 5
sorted_population = sorted(enumerate(fitness_scores), key=lambda x: x[1], reverse=True)
selected = [initial_population[idx] for idx, _ in sorted_population[:top_N]]
print("\nTop selected chromosomes (fitness):")

def crossover(parent1, parent2):
    # Find a common node to split on (excluding depot start/end)
    common_nodes = set(a['dst']['id'] for a in parent1[1:-1]) & set(a['dst']['id'] for a in parent2[1:-1])
    if not common_nodes:
        return parent1.copy(), parent2.copy()  # No crossover possible

    split_node_id = random.choice(list(common_nodes))

    def split_route(route, node_id):
        idx = next((i for i, arc in enumerate(route) if arc['dst']['id'] == node_id), None)
        return route[:idx + 1], route[idx + 1:]

    p1_head, p1_tail = split_route(parent1, split_node_id)
    p2_head, p2_tail = split_route(parent2, split_node_id)

    child1 = p1_head + p2_tail
    child2 = p2_head + p1_tail

    # Ensure both children are valid (start to end node)
    if child1[0]['src']['id'] == start_node_id and child1[-1]['dst']['id'] == end_node_id:
        pass
    else:
        child1 = parent1.copy()

    if child2[0]['src']['id'] == start_node_id and child2[-1]['dst']['id'] == end_node_id:
        pass
    else:
        child2 = parent2.copy()

    return child1, child2

for i, chrom in enumerate(selected):
    fit = evaluate_fitness(chrom)
    print(f"Selected #{i + 1}: Fitness = {fit}")

for idx, chrom in enumerate(initial_population):
    fitness = evaluate_fitness(chrom)
    fitness_scores.append((idx + 1, fitness))
    print(f"Chromosome #{idx + 1}: Fitness = {fitness}")

best = max(fitness_scores, key=lambda x: x[1])
print(f"\nBest Chromosome is #{best[0]} with Fitness = {best[1]}")

# ---------------------------------------------------------------
    


{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 2, 'dst_loc': 'ATB', 'dst_time': 720, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 3, 'dst_loc': 'HSK', 'dst_time': 720, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 4, 'dst_loc': 'ATB', 'dst_time': 780, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 6, 'dst_loc': 'HSK', 'dst_time': 780, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 5, 'dst_loc': 'ATB', 'dst_time': 810, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 7, 'dst_loc': 'HSK', 'dst_time': 810, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 8, 'dst_loc': 'ATB', 'dst_time': 840, 'cost': 40, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'dst_id': 9, 'dst_loc': 'HSK', 'dst_time': 840, 'cost': 10, 'N_bus': 150}
{'src_id': 1, 'src_loc': 'X', 'src_time': 0, 'ds